In [2]:
import warnings
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    average_precision_score,
    classification_report
)

warnings.filterwarnings('ignore')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')

PyTorch: 2.11.0+cu130
CUDA disponible: True


In [3]:
BASE_DATA_DIR = '../data'
OUTPUT_DIR = '../data/processed'
MODELS_DIR = '../models'
RESULTS_DIR = '../results'

SEQ_LEN = 4 # Horas de historia por secuencia (reducido para capturar mas sepsis con onset temprano)
PREDICTION_HORIZON_H = 6 # Predice onset en las proximas 6 horas
MIN_HOURS = SEQ_LEN # Minimo de horas con datos para incluir estancia

# Hiperparametros
BATCH_SIZE = 256
EPOCHS = 150
LR = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.4
PATIENCE = 20
WARMUP_EPOCHS = 2 # Epocas de calentamiento lineal del LR

RANDOM_STATE_SEED = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')

Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

Dispositivo: cuda


In [4]:
features_df = pd.read_parquet(f'{OUTPUT_DIR}/features_df.parquet')
cohort_df = pd.read_parquet(f'{OUTPUT_DIR}/cohort_df.parquet')

FEATURE_COLS = [
    col for col in features_df.columns
    if col not in ['stay_id', 'hour_bucket', 'Bands'] # Quitamos Bands por 99.7% NaN
]

N_FEATURES = len(FEATURE_COLS)

# Convierte str a datetime
cohort_df['intime'] = pd.to_datetime(cohort_df['intime'])
cohort_df['sepsis_onset'] = pd.to_datetime(cohort_df['sepsis_onset'])

# Calcular onset_h: Horas desde intime hasta el onset (solo para sepsis)
cohort_df['onset_h'] = np.where(
    cohort_df['label'] == 1,
    (cohort_df['sepsis_onset'] - cohort_df['intime']).dt.total_seconds() / 3600,
    np.nan
)

# Mapeamos las horas por estancia y el resultado (sepsis / no sepsis)
ONSET_H_MAP = cohort_df.set_index('stay_id')['onset_h'].to_dict()
LABEL_MAP = cohort_df.set_index('stay_id')['label'].to_dict()

# Cargar SOFA horario para target auxiliar de multi-task 
sofa_hourly = pd.read_parquet(f'{OUTPUT_DIR}/sofa_hourly.parquet')
sofa_hourly['hour_bucket'] = sofa_hourly['h_since_intime'].astype(int)
print(f'SOFA HOURLY: {len(sofa_hourly)}')

# Lookup (stay_id, hour_bucket) - SOFA. Para controles devolvera NaN
SOFA_LOOKUP = sofa_hourly.set_index(['stay_id', 'hour_bucket'])['sofa'].to_dict()

print(f'Estancias totales: {cohort_df["stay_id"].nunique():,}')
print(f'    Sepsis (label = 1): {(cohort_df["label"] == 1).sum():,}')
print(f'    Control (label = 0): {(cohort_df["label"] == 0).sum():,}')
print()
print(f'Features ({N_FEATURES}): {FEATURE_COLS}')

SOFA HOURLY: 5203081
Estancias totales: 74,754
    Sepsis (label = 1): 37,377
    Control (label = 0): 37,377

Features (26): ['Arterial Blood Pressure mean', 'GCS - Motor Response', 'GCS - Verbal Response', 'Heart Rate', 'Non Invasive Blood Pressure diastolic', 'Non Invasive Blood Pressure systolic', 'O2 saturation pulseoxymetry', 'PEEP set', 'Respiratory Rate', 'Temperature Celsius', 'Bicarbonate', 'Bilirubin, Total', 'Creatinine', 'Glucose', 'Lactate', 'Platelet Count', 'Urea Nitrogen', 'White Blood Cells', 'pH', 'pO2', 'urine_output_ml', 'vasopressor_active', 'mechanical_ventilation', 'age', 'gender_enc', 'charlson_score']


In [5]:
stays_coverage = features_df.groupby('stay_id')['hour_bucket'].count()
valid_stay_ids = stays_coverage[stays_coverage >= MIN_HOURS].index

stay_labels_v = cohort_df[cohort_df['stay_id'].isin(valid_stay_ids)][
    ['stay_id', 'label', 'subject_id']
].reset_index(drop=True)

all_stays = stay_labels_v['stay_id'].values
all_labels = stay_labels_v['label'].values
all_subjects = stay_labels_v['subject_id'].values

first_shuffle_split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE_SEED)
tv_idx, test_idx = next(
    first_shuffle_split.split(all_stays, all_labels, groups=all_subjects)
)

second_shuffle_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE_SEED)
tr_idx, val_idx = next(
    second_shuffle_split.split(all_stays[tv_idx], all_labels[tv_idx], groups=all_subjects[tv_idx])
)

train_stays = all_stays[tv_idx][tr_idx]
val_stays = all_stays[tv_idx][val_idx]
test_stays = all_stays[test_idx]

stay_to_subject = cohort_df.set_index('stay_id')['subject_id']
train_sub = set(stay_to_subject[stay_to_subject.index.isin(train_stays)])
val_sub = set(stay_to_subject[stay_to_subject.index.isin(val_stays)])
test_sub = set(stay_to_subject[stay_to_subject.index.isin(test_stays)])

print(f'Train: {len(train_stays):,} estancias | {len(train_sub):,} pacientes')
print(f'Validation: {len(val_stays):,} estancias | {len(val_sub):,} pacientes')
print(f'Test: {len(test_stays):,} estancias | {len(test_sub):,} pacientes')

Train: 44,794 estancias | 32,220 pacientes
Validation: 14,889 estancias | 10,740 pacientes
Test: 14,871 estancias | 10,740 pacientes


In [6]:
def build_sequences(
    features_df,
    stay_ids,
    feature_cols,
    onset_h_map,
    label_map,
    sofa_lookup = None,
    seq_len = SEQ_LEN,
    horizon_h = PREDICTION_HORIZON_H,
):
    """
    Genera secuencias para prediccion rodante (rolling prediction).

    Para cada estancia y cada hora t con datos previos al onset:
        - X[i]: Ultimas seq_len horas de features hasta t (seq_len, F)
        - y[i]: tiene 1 si onset ocurre en (t, t + horizon_h), 0 si no
        - stay_id[i]: stay_id (para garantizar split sin leakage)
        - lead[i]: horas hasta el onset (NaN para controles)

    hour_bucket debe ser >= 0 (horas desde intime).
    """

    features_grouped_by_stay = dict(tuple(
        features_df[features_df['stay_id'].isin(stay_ids)].groupby('stay_id')
    ))

    X_list, y_list, sid_list, lead_list, sofa_list = [], [], [], [], []

    for stay_id in stay_ids:
        if stay_id not in features_grouped_by_stay:
            continue

        df_sorted_by_stay = features_grouped_by_stay[stay_id].sort_values('hour_bucket')
        is_sepsis = label_map.get(stay_id, 0) # Comprueba si es una estancia que acaba en sepsis, devuelve 0 por defecto si es un valor nulo
        onset_hour = onset_h_map.get(stay_id)

        # Filtro clinico, descarta los casos de sepsis que llegan en deterioro
        MIN_ONSET_H = 6
        if is_sepsis and onset_hour is not None and not pd.isna(onset_hour) and onset_hour < MIN_ONSET_H:
            continue
        
        # Excluye horas tras el onset para casos de sepsis
        # Para controles, capa LOS a 168h (7 dias) para evitar estancias largas
        CONTROL_MAX_LOS_H = 168
        if is_sepsis and onset_hour is not None and not pd.isna(onset_hour):
            max_hour = int(onset_hour) - 1
        else:
            max_hour = min(int(df_sorted_by_stay['hour_bucket'].max()), CONTROL_MAX_LOS_H - 1)

        df_sorted_by_stay = df_sorted_by_stay[df_sorted_by_stay['hour_bucket'].between(0, max_hour)]
        if df_sorted_by_stay['hour_bucket'].nunique() < seq_len:
            continue
        
        # Grid denso (hora 0 ... max_hour) con NaN donde no hay medicion
        grid = np.full((max_hour + 1, N_FEATURES), np.nan, dtype=np.float32)
        hb_values = df_sorted_by_stay['hour_bucket'].astype(int).values
        feat_values = df_sorted_by_stay[feature_cols].values.astype(np.float32)
        valid_mask = (hb_values >= 0) & (hb_values <= max_hour)
        grid[hb_values[valid_mask]] = feat_values[valid_mask]

        if len(grid) < seq_len:
            continue

        # Ventana deslizante vectorizada -> (n_windows, seq_len, N_FEATURES)
        windows = sliding_window_view(grid, (seq_len, N_FEATURES)).reshape(-1, seq_len, N_FEATURES)
        num_windows = len(windows)
        end_hour = np.arange(seq_len - 1, seq_len - 1 + num_windows, dtype=np.float32)

        # Etiquetas rodantes
        if is_sepsis and onset_hour is not None and not pd.isna(onset_hour):
            hour_to_onset = float(onset_hour) - end_hour
            labels = ((hour_to_onset > 0) & (hour_to_onset <= horizon_h)).astype(np.float32)
            leads = hour_to_onset
        else:
            labels = np.zeros(num_windows, dtype=np.float32)
            leads = np.full(num_windows, np.nan, dtype=np.float32)

        # Subsampling para controles: max 30 ventanas/estancia repartidas
        # De forma uniforme a lo largo de la estancia para preservar diversidad temporal
        MAX_WINDOWS_PER_CONTROL = 30
        if not is_sepsis and num_windows > MAX_WINDOWS_PER_CONTROL:
            keep_idx = np.linspace(0, num_windows - 1, MAX_WINDOWS_PER_CONTROL).astype(int)
            windows = windows[keep_idx]
            labels = labels[keep_idx]
            leads = leads[keep_idx]

        # SOFA en el momento de prediccion (end_hour) - target auxiliar
        # Es seguro: SOFA en end_hour es info disponible AL ALERTAR, no info futura
        if sofa_lookup is not None:
            sofa_window = np.array([sofa_lookup.get((stay_id, int(hour)), np.nan) for hour in end_hour], dtype=np.float32)
        else:
            sofa_window = np.full(num_windows, np.nan, dtype=np.float32)

        X_list.append(windows)
        y_list.append(labels)
        sid_list.extend([stay_id] * len(windows))
        lead_list.append(leads)
        sofa_list.append(sofa_window)

    if not X_list:
        empty = np.empty((0, seq_len, N_FEATURES), dtype=np.float32)
        return empty, np.array([]), np.array([]), np.array([]), np.array([])
    
    return (
        np.concatenate(X_list, axis=0),
        np.concatenate(y_list, axis=0),
        np.array(sid_list),
        np.concatenate(lead_list, axis=0),
        np.concatenate(sofa_list, axis=0),
    )


In [7]:
X_train, y_train, sids_train, leads_train, sofa_train = build_sequences(
    features_df = features_df,
    stay_ids = train_stays,
    feature_cols = FEATURE_COLS,
    onset_h_map = ONSET_H_MAP,
    label_map = LABEL_MAP,
    sofa_lookup = SOFA_LOOKUP,
)
X_val, y_val, sids_val, leads_val, sofa_val = build_sequences(
    features_df = features_df,
    stay_ids = val_stays,
    feature_cols = FEATURE_COLS,
    onset_h_map = ONSET_H_MAP,
    label_map = LABEL_MAP,
    sofa_lookup = SOFA_LOOKUP,
)
X_test, y_test, sids_test, leads_test, sofa_test = build_sequences(
    features_df = features_df,
    stay_ids = test_stays,
    feature_cols = FEATURE_COLS,
    onset_h_map = ONSET_H_MAP,
    label_map = LABEL_MAP,
    sofa_lookup = SOFA_LOOKUP,
)

print(f'X_train: {X_train.shape} | positivos: {y_train.mean():.2%}')
print(f'X_val: {X_val.shape} | positivos: {y_val.mean():.2%}')
print(f'X_test: {X_test.shape} | positivos: {y_test.mean():.2%}')


X_train: (712829, 4, 26) | positivos: 4.81%
X_val: (236776, 4, 26) | positivos: 4.84%
X_test: (236872, 4, 26) | positivos: 4.64%


In [8]:
# Subsampling estratificado de negativos en train (ratio 5:1 neg:pos)
# Val/test se dejan completos para preservar la prevalencia real en métricas
NEG_POS_RATIO = 5
pos_idx = np.where(y_train == 1)[0]
neg_idx = np.where(y_train == 0)[0]
n_neg_keep = min(len(neg_idx), len(pos_idx) * NEG_POS_RATIO)
_rng = np.random.default_rng(RANDOM_STATE_SEED)
neg_keep = _rng.choice(neg_idx, size=n_neg_keep, replace=False)
keep = np.sort(np.concatenate([pos_idx, neg_keep]))
X_train  = X_train[keep]
y_train = y_train[keep]
sids_train = sids_train[keep]
leads_train = leads_train[keep]
sofa_train = sofa_train[keep]
print(f'Tras subsampling — X_train: {X_train.shape} | positivos: {y_train.mean():.2%}')

Tras subsampling — X_train: (205680, 4, 26) | positivos: 16.67%


In [9]:
def apply_forward_fill_vectorized(X, max_gap=24):
    """
    Forward fill vectorizado sobre el eje temporal (axis=1) con tope de propagación.

    max_gap: máximo de horas que un valor puede propagarse hacia adelante antes
    de volver a NaN (por defecto 24h). Evita arrastrar labs medidos hace varios
    días como si fueran actuales.
    """
    n, T, F = X.shape
    X_ff = X.copy()
    arange_T = np.arange(T)
    for j in range(F):
        arr  = X_ff[:, :, j]                          # (n, T)
        mask = ~np.isnan(arr)                          # True donde hay valor real
        idx  = np.where(mask, arange_T, 0)            # índice del último válido, o 0
        np.maximum.accumulate(idx, axis=1, out=idx)   # propagar último índice válido
        gap  = arange_T[None, :] - idx                # distancia (horas) al último valor real
        filled = arr[np.arange(n)[:, None], idx]
        # Si el gap supera max_gap, el valor propagado se descarta (vuelve a NaN)
        too_old = gap > max_gap
        filled = np.where(too_old, np.nan, filled)
        X_ff[:, :, j] = filled
    return X_ff


# Calcular máscaras ANTES del forward fill (refleja mediciones reales)
M_train = (~np.isnan(X_train)).astype(np.float32)
M_val   = (~np.isnan(X_val)).astype(np.float32)
M_test  = (~np.isnan(X_test)).astype(np.float32)

X_train_ff = apply_forward_fill_vectorized(X_train)
X_val_ff   = apply_forward_fill_vectorized(X_val)
X_test_ff  = apply_forward_fill_vectorized(X_test)

pct_nan_antes   = np.isnan(X_train).mean()
pct_nan_despues = np.isnan(X_train_ff).mean()
print(f'NaN antes del ffill:   {pct_nan_antes:.1%}')
print(f'NaN despues del ffill: {pct_nan_despues:.1%}')

NaN antes del ffill:   56.9%
NaN despues del ffill: 47.8%


In [10]:
def compute_deltas(X_ff, M_original=None):
    """
    Diferencias temporales (deltas) sobre features forward-filleadas.

    delta[t] = X_ff[t] - X_ff[t-1]  para t > 0, delta[0] = 0.
    Los NaN en la diferencia (alguno de los dos timesteps sin valor tras ffill)
    se sustituyen por 0 antes de aplicar la mascara, ya que en numpy NaN*0 = NaN
    y propagaria NaN al LSTM.
    Si se proporciona M_original, el delta se anula cuando alguno de los dos
    timesteps consecutivos no tuvo medicion real.
    """
    deltas = np.zeros_like(X_ff)
    diff = X_ff[:, 1:, :] - X_ff[:, :-1, :]
    # NaN en la diferencia → 0 (evita NaN * 0 = NaN al enmascarar)
    deltas[:, 1:, :] = np.where(np.isnan(diff), 0.0, diff)

    if M_original is not None:
        # Delta valido solo si t-1 y t tienen medicion real
        both_measured = M_original[:, 1:, :] * M_original[:, :-1, :]
        deltas[:, 1:, :] *= both_measured

    return deltas.astype(np.float32)


deltas_train = compute_deltas(X_train_ff, M_train)
deltas_val   = compute_deltas(X_val_ff,   M_val)
deltas_test  = compute_deltas(X_test_ff,  M_test)

nonzero_pct = (deltas_train != 0).mean()
print(f'Deltas train shape: {deltas_train.shape}')
print(f'Deltas no nulos (cambios reales medidos): {nonzero_pct:.2%}')
print(f'NaN en deltas_train: {np.isnan(deltas_train).sum()}')


Deltas train shape: (205680, 4, 26)
Deltas no nulos (cambios reales medidos): 11.39%
NaN en deltas_train: 0


In [11]:
def apply_missingness_mask(X, train_medians=None, external_mask=None, external_deltas=None):
    """
    Estrategia de imputación con missingness mask y deltas opcionales:
    1. Construye máscara binaria M: 1 = valor real, 0 = imputado.
    2. Rellena NaN con la mediana global del training (valor neutro).
    3. Concatena [X_imputed | deltas | M] → shape (n, T, 3*F) si se pasan deltas,
       o [X_imputed | M] → shape (n, T, 2*F) si no.

    external_mask: máscara pre-computada desde el tensor original con NaN.
    external_deltas: deltas pre-computados desde apply_forward_fill (opcional).
    """

    _, _, F = X.shape

    M = external_mask if external_mask is not None else (~np.isnan(X)).astype(np.float32)

    X_imp = X.copy()

    if train_medians is None:
        flat = X_imp.reshape(-1, F)
        train_medians = np.nanmedian(flat, axis=0)
        train_medians = np.where(np.isnan(train_medians), 0.0, train_medians)

    for j in range(F):
        mask_nan = np.isnan(X_imp[:, :, j])
        X_imp[:, :, j][mask_nan] = train_medians[j]

    parts = [X_imp]
    if external_deltas is not None:
        parts.append(external_deltas)
    parts.append(M)

    X_out = np.concatenate(parts, axis=2).astype(np.float32)
    return X_out, train_medians


X_train_masked, train_medians = apply_missingness_mask(
    X_train_ff, external_mask=M_train, external_deltas=deltas_train
)
X_val_masked, _ = apply_missingness_mask(
    X_val_ff, train_medians, external_mask=M_val, external_deltas=deltas_val
)
X_test_masked, _ = apply_missingness_mask(
    X_test_ff, train_medians, external_mask=M_test, external_deltas=deltas_test
)

print(f'Shape tras aplicar mask + deltas: {X_train_masked.shape} → input_size al LSTM: {X_train_masked.shape[2]}')


Shape tras aplicar mask + deltas: (205680, 4, 78) → input_size al LSTM: 78


In [12]:
# Estructura del tensor: [valores (F) | deltas (F) | mascaras (F)] → F_total = 3*F
# Solo valores y deltas se normalizan; las mascaras ya son binarias {0,1}
n_tr, T, F_total = X_train_masked.shape
F = F_total // 3

scaler = StandardScaler()

def scale_masked(X, scaler, F, fit=False):
    n, T, F_total = X.shape
    has_deltas = (F_total == 3 * F)

    X_vals = X[:, :, :F].reshape(-1, F)
    if fit:
        X_vals_scaled = scaler.fit_transform(X_vals)
    else:
        X_vals_scaled = scaler.transform(X_vals)
    X_vals_scaled = X_vals_scaled.reshape(n, T, F).astype(np.float32)

    if has_deltas:
        # Deltas se normalizan solo por la desviacion estandar (no se resta media:
        # delta=0 significa "sin cambio" y debe mapearse a 0 tras normalizacion)
        delta_std = np.where(scaler.scale_ > 0, scaler.scale_, 1.0)
        X_deltas = X[:, :, F:2*F].reshape(-1, F)
        X_deltas_scaled = (X_deltas / delta_std).reshape(n, T, F).astype(np.float32)
        X_mask = X[:, :, 2*F:]
        return np.concatenate([X_vals_scaled, X_deltas_scaled, X_mask], axis=2).astype(np.float32)
    else:
        X_mask = X[:, :, F:]
        return np.concatenate([X_vals_scaled, X_mask], axis=2).astype(np.float32)


X_train_norm = scale_masked(X_train_masked, scaler, F, fit=True)
X_val_norm   = scale_masked(X_val_masked,   scaler, F, fit=False)
X_test_norm  = scale_masked(X_test_masked,  scaler, F, fit=False)

N_INPUT = X_train_norm.shape[2]  # 3*F = valores + deltas + mascaras
print(f'Input al LSTM: {N_INPUT}  (shape: {X_train_norm.shape})')
print(f'  {F} valores normalizados | {F} deltas normalizados | {F} mascaras binarias')
print(f'Mascaras — rango: [{X_train_norm[:,:,2*F:].min():.0f}, {X_train_norm[:,:,2*F:].max():.0f}]')


Input al LSTM: 78  (shape: (205680, 4, 78))
  26 valores normalizados | 26 deltas normalizados | 26 mascaras binarias
Mascaras — rango: [0, 1]


In [13]:
class SepsisDataset(Dataset):
    """Clase que alimenta al modelo durante el entrenamiento"""
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = SepsisDataset(X_train_norm, y_train)
val_ds = SepsisDataset(X_val_norm, y_val)
test_ds = SepsisDataset(X_test_norm, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# pos_weight: con predicción rodante los positivos son minoritarios (~5-15%)
n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos
pos_weight_val = n_neg / max(n_pos, 1)

N_INPUT = X_train_norm.shape[2]
print(f'Input shape: {X_train_norm.shape} → N_INPUT: {N_INPUT}')
print(f'Train batches: {len(train_loader)} x {BATCH_SIZE}')
print(f'Positivos: {n_pos:,} ({n_pos/len(y_train):.1%}) | Negativos: {n_neg:,} | pos_weight: {pos_weight_val:.2f}')

Input shape: (205680, 4, 78) → N_INPUT: 78
Train batches: 803 x 256
Positivos: 34,280 (16.7%) | Negativos: 171,400 | pos_weight: 5.00


In [14]:
class Bi_LSTM_Sepsis(nn.Module):
    """Bi_LSTM_Sepsis con mecanismo de atención sobre el eje temporal.

    Reemplaza el avg+max pooling fijo con atención aprendida:
    el modelo aprende a ponderar cada timestep según su relevancia para la predicción.
    La atención es especialmente útil en series clínicas donde las horas más cercanas
    al momento de predicción suelen ser más informativas.
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int = HIDDEN_SIZE,
        num_layers: int = NUM_LAYERS,
        dropout: float = DROPOUT
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )

        lstm_out_size = hidden_size * 2  # bidireccional: concat forward + backward

        # Atención sobre el eje temporal: una puntuación escalar por timestep
        self.attn = nn.Linear(lstm_out_size, 1)

        self.bn1 = nn.BatchNorm1d(lstm_out_size)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(lstm_out_size, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.lstm(x)                                # (batch, T, hidden*2)

        # Atención: softmax sobre la dimensión temporal → suma ponderada
        attn_weights = torch.softmax(self.attn(out), dim=1)  # (batch, T, 1)
        context = (attn_weights * out).sum(dim=1)             # (batch, hidden*2)

        context = self.bn1(context)
        context = self.dropout(context)
        context = self.relu(self.fc1(context))
        return self.fc2(context).squeeze(-1)


model = Bi_LSTM_Sepsis(input_size=N_INPUT).to(DEVICE)
print(model)

Bi_LSTM_Sepsis(
  (lstm): LSTM(78, 128, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  (attn): Linear(in_features=256, out_features=1, bias=True)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.4, inplace=False)
  (fc1): Linear(in_features=256, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)


In [15]:
class FocalLoss(nn.Module):
    """Focal Loss para clasificación binaria con desbalance extremo.

    L = -α (1-p)^γ log(p)  para positivos
    L = -(1-α) p^γ log(1-p) para negativos
    """
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        # BCE estándar sin reducción
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )
        # p_t: probabilidad del modelo de la clase verdadera
        p   = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        # alpha_t: alpha para positivos, 1-alpha para negativos
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal = alpha_t * (1 - p_t) ** self.gamma * bce
        return focal.mean()

In [16]:
def train_model(model, model_name, pos_weight_value=None):
    # Focal loss reemplaza BCE+pos_weight para imbalance extremo (1.46% positivos)
    # pos_weight_value se mantiene en la firma por compatibilidad pero queda ignorado
    criterion = FocalLoss(alpha=0.25, gamma=1.0)  # gamma=1 para mejor calibración (mejora AUPRC)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # Scheduler sobre val_auroc (mode=max) en lugar de val_loss:
    # val_loss es ruidosa epoch a epoch y dispara reducciones prematuras;
    # val_auroc es la metrica objetivo real y tiene menor varianza entre epocas.
    # patience=7 da al modelo suficiente margen antes de cada reduccion de LR.
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=7, factor=0.5
    )

    history          = {'train_loss': [], 'val_loss': [], 'val_auroc': []}
    best_val_auroc   = 0.0
    patience_counter = 0
    best_path        = f'{MODELS_DIR}/{model_name}.pt'

    for epoch in range(1, EPOCHS + 1):

        # Warmup lineal del LR durante las primeras WARMUP_EPOCHS épocas
        if epoch <= WARMUP_EPOCHS:
            warmup_lr = LR * epoch / WARMUP_EPOCHS
            for g in optimizer.param_groups:
                g['lr'] = warmup_lr

        # ── Fase entrenamiento ──
        model.train()
        train_loss    = 0.0
        train_samples = 0

        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss    += loss.item() * len(y_b)
            train_samples += len(y_b)

        train_loss /= train_samples

        # ── Fase validación ──
        model.eval()
        val_loss  = 0.0
        val_probs, val_true = [], []

        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
                logits = model(X_b)
                val_loss += criterion(logits, y_b).item() * len(y_b)
                val_probs.extend(torch.sigmoid(logits).cpu().numpy())
                val_true.extend(y_b.cpu().numpy())

        val_loss  /= len(val_loader.dataset)
        val_auroc  = roc_auc_score(val_true, val_probs) if len(set(val_true)) > 1 else 0.0

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auroc'].append(val_auroc)

        old_lr = optimizer.param_groups[0]['lr']
        if epoch > WARMUP_EPOCHS:
            scheduler.step(val_auroc)
        new_lr = optimizer.param_groups[0]['lr']
        if new_lr < old_lr:
            print(f'--- Tasa de aprendizaje reducida a: {new_lr:.2e} ---')

        # Early stopping y checkpoint sobre val_auroc
        if val_auroc > best_val_auroc:
            best_val_auroc   = val_auroc
            patience_counter = 0
            torch.save(model.state_dict(), best_path)
        else:
            patience_counter += 1

        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
              f'Val AUROC: {val_auroc:.4f} | LR: {new_lr:.2e} | P: {patience_counter}')

        if patience_counter >= PATIENCE:
            print(f'Early stopping en epoch {epoch}. AUROC no mejoró en {PATIENCE} épocas.')
            break

    print(f'\nMejor val_auroc: {best_val_auroc:.4f}')
    return {
        'history': history,
        'path': best_path,
        'model': model,
        'pos_weight': None
    }

models = {}

model_name = 'best_lstm_attention'
models[model_name] = train_model(model, model_name, n_neg / max(n_pos, 1))


Epoch   1/150 | Train Loss: 0.0741 | Val Loss: 0.0392 | Val AUROC: 0.7644 | LR: 5.00e-04 | P: 0
Epoch   2/150 | Train Loss: 0.0714 | Val Loss: 0.0381 | Val AUROC: 0.7645 | LR: 1.00e-03 | P: 0
Epoch   3/150 | Train Loss: 0.0708 | Val Loss: 0.0400 | Val AUROC: 0.7649 | LR: 1.00e-03 | P: 0
Epoch   4/150 | Train Loss: 0.0704 | Val Loss: 0.0387 | Val AUROC: 0.7694 | LR: 1.00e-03 | P: 0
Epoch   5/150 | Train Loss: 0.0701 | Val Loss: 0.0381 | Val AUROC: 0.7726 | LR: 1.00e-03 | P: 0
Epoch   6/150 | Train Loss: 0.0700 | Val Loss: 0.0413 | Val AUROC: 0.7754 | LR: 1.00e-03 | P: 0
Epoch   7/150 | Train Loss: 0.0699 | Val Loss: 0.0401 | Val AUROC: 0.7734 | LR: 1.00e-03 | P: 1
Epoch   8/150 | Train Loss: 0.0697 | Val Loss: 0.0391 | Val AUROC: 0.7743 | LR: 1.00e-03 | P: 2
Epoch   9/150 | Train Loss: 0.0696 | Val Loss: 0.0412 | Val AUROC: 0.7743 | LR: 1.00e-03 | P: 3
Epoch  10/150 | Train Loss: 0.0695 | Val Loss: 0.0397 | Val AUROC: 0.7737 | LR: 1.00e-03 | P: 4
Epoch  11/150 | Train Loss: 0.0696 | Val

In [18]:
def show_model_metrics_optimized(model, model_name, model_path, leads=None):
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    model.eval()

    test_probs, test_true = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            logits = model(X_b.to(DEVICE))
            test_probs.extend(torch.sigmoid(logits).cpu().numpy())
            test_true.extend(y_b.numpy())

    test_probs = np.array(test_probs)
    test_true  = np.array(test_true)

    fpr, tpr, thresholds = roc_curve(test_true, test_probs)
    best_thresh = thresholds[np.argmax(tpr - fpr)]
    preds = (test_probs >= best_thresh).astype(int)

    auroc = roc_auc_score(test_true, test_probs)
    auprc = average_precision_score(test_true, test_probs)

    print(f'=== {model_name} (Umbral Opt: {best_thresh:.3f}) ===')
    print(f'AUROC: {auroc:.4f}  |  AUPRC: {auprc:.4f}')
    print(classification_report(test_true, preds, target_names=['No alerta', 'Alerta']))

    # Curva de operación: precisión a recall objetivo y precisión @ top-k
    print('\n--- Curva de operación clínica ---')
    for target_recall in [0.30, 0.50, 0.70]:
        # Buscar el umbral más alto que aún garantiza recall >= target
        eligible = np.where(tpr >= target_recall)[0]
        if len(eligible) == 0:
            print(f'  Recall >= {target_recall:.0%} → no alcanzable')
            continue
        i = eligible[0]  # primer punto que cruza el target (umbral más alto)
        thr  = thresholds[i]
        sel  = test_probs >= thr
        prec = (sel & (test_true == 1)).sum() / max(sel.sum(), 1)
        rec  = tpr[i]
        print(f'  Recall ≥ {target_recall:.0%} → umbral={thr:.3f}  precisión={prec:.3f}  recall_real={rec:.3f}')

    # Precision @ top-k: las k alertas con mayor probabilidad
    print('\n--- Precision @ top-k alertas ---')
    for top_pct in [0.01, 0.05, 0.10]:
        k = max(int(len(test_probs) * top_pct), 1)
        top_k = np.argsort(-test_probs)[:k]
        prec_k = test_true[top_k].mean()
        print(f'  Top {top_pct:.0%} ({k:,} alertas) → precisión={prec_k:.3f}')

    # AUPRC normalizado por prevalencia (1.0 = nivel aleatorio)
    prevalence = test_true.mean()
    auprc_lift = auprc / prevalence if prevalence > 0 else float('nan')
    print(f'\nPrevalencia test: {prevalence:.2%}  |  AUPRC lift sobre baseline: {auprc_lift:.2f}×')

    # Tiempo medio de alerta temprana (metrica clave para sistema de alerta)
    if leads is not None:
        leads_arr = np.array(leads)
        # Verdaderos positivos: label=1 (onset en horizonte) y modelo predice 1
        tp_mask = (test_true == 1) & (preds == 1) & ~np.isnan(leads_arr)
        fn_mask = (test_true == 1) & (preds == 0) & ~np.isnan(leads_arr)
        if tp_mask.any():
            print(f'\nTiempo de alerta (TP): {leads_arr[tp_mask].mean():.1f}h antes del onset '
                  f'(mediana: {np.median(leads_arr[tp_mask]):.1f}h)')
        print(f'Alarmas emitidas (TP): {tp_mask.sum():,} | Perdidas (FN): {fn_mask.sum():,}')

    return auroc, auprc

for model_name, values in models.items():
    show_model_metrics_optimized(
        model=values['model'],
        model_name=model_name,
        model_path=values['path'],
        leads=leads_test,
    )

=== best_lstm_attention (Umbral Opt: 0.172) ===
AUROC: 0.7809  |  AUPRC: 0.1702
              precision    recall  f1-score   support

   No alerta       0.98      0.70      0.82    225875
      Alerta       0.11      0.73      0.18     10997

    accuracy                           0.70    236872
   macro avg       0.54      0.71      0.50    236872
weighted avg       0.94      0.70      0.79    236872


--- Curva de operación clínica ---
  Recall ≥ 30% → umbral=0.302  precisión=0.205  recall_real=0.300
  Recall ≥ 50% → umbral=0.237  precisión=0.149  recall_real=0.500
  Recall ≥ 70% → umbral=0.178  precisión=0.110  recall_real=0.700

--- Precision @ top-k alertas ---
  Top 1% (2,368 alertas) → precisión=0.318
  Top 5% (11,843 alertas) → precisión=0.230
  Top 10% (23,687 alertas) → precisión=0.179

Prevalencia test: 4.64%  |  AUPRC lift sobre baseline: 3.67×

Tiempo de alerta (TP): 3.2h antes del onset (mediana: 3.0h)
Alarmas emitidas (TP): 7,973 | Perdidas (FN): 3,024
